# 📘 분할표 검정

**분할표**(Contingency Table)는 범주형 데이터의 빈도를 정리한 표입니다.
예: 버튼 색(빨강/파랑)과 클릭 여부(함/안 함)의 관계

**카이제곱 검정**(Chi-squared test)은 분할표에서 두 범주형 변수 간의
독립성(관련성이 없는지)을 검정합니다.

**학습 목표:**
- 분할표의 개념과 작성법
- 카이제곱 통계량의 의미
- `stats.chi2_contingency()` 사용법
- p값과 카이제곱분포의 관계

## 1. 분할표란?

**분할표**(Cross table)는 두 범주형 변수의 빈도를 교차 정리한 표입니다.

예: 웹사이트 버튼 색과 클릭 여부

|  | 클릭 함 | 클릭 안 함 | 합계 |
|--|---------|-----------|------|
| 파랑 | 20 | 230 | 250 |
| 빨강 | 10 | 40 | 50 |
| 합계 | 30 | 270 | 300 |

> 💡 분할표에서 확인할 것: "버튼 색에 따라 클릭률이 다른가?"
> 빨강 버튼의 클릭률(10/50=20%)이 파랑 버튼(20/250=8%)보다 높습니다.
> 이 차이가 우연인지 통계적으로 유의미한지 검정합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  라이브러리 임포트 + 데이터 로드            │
# └─────────────────────────────────────────┘

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 클릭 데이터 불러오기
click_data = pd.read_csv("click_data.csv")
print("=== 원본 클릭 데이터 ===")
print(click_data)

# 분할표로 변환
cross = pd.pivot_table(
    data=click_data,
    values="freq",
    aggfunc="sum",
    index="color",
    columns="click"
)
print(f"\n=== 분할표 ===")
print(cross)

# 클릭률 계산
for color in cross.index:
    total = cross.loc[color].sum()
    click_rate = cross.loc[color, 'click'] / total * 100
    print(f"{color} 버튼 클릭률: {click_rate:.1f}% ({cross.loc[color, 'click']}/{total})")

## 2. 카이제곱 통계량과 p값

**카이제곱 검정**의 핵심 아이디어:
1. **기대빈도**: 두 변수가 독립일 때 예상되는 빈도
2. **카이제곱 통계량**: 관측빈도와 기대빈도의 차이를 합산

$$\chi^2 = \sum \frac{(관측빈도 - 기대빈도)^2}{기대빈도}$$

- χ²이 크면 → 관측과 기대가 많이 다름 → 독립이 아님 (관련 있음)
- χ²이 작으면 → 관측과 기대가 비슷 → 독립일 가능성 높음

> 💡 p값은 카이제곱분포에서 계산됩니다.
> p < 0.05이면 "두 변수는 독립이 아니다"라고 결론냅니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  카이제곱 검정 수행                       │
# │  stats.chi2_contingency() 사용           │
# │  correction=False: Yates 보정 없이        │
# └─────────────────────────────────────────┘

# 카이제곱 검정
result = stats.chi2_contingency(cross, correction=False)
chi2 = result[0]      # 카이제곱 통계량
p_value = result[1]   # p값
dof = result[2]       # 자유도
expected = result[3]  # 기대빈도

print("=== 카이제곱 검정 결과 ===")
print(f"카이제곱 통계량 (χ²): {chi2:.4f}")
print(f"p값: {p_value:.6f}")
print(f"자유도: {dof}")
print(f"\n=== 기대빈도 (독립이라고 가정했을 때) ===")
expected_df = pd.DataFrame(expected, index=cross.index, columns=cross.columns)
print(expected_df.round(2))

# 판정
alpha = 0.05
if p_value < alpha:
    print(f"\np값 {p_value:.6f} < {alpha} → 귀무가설 기각!")
    print("→ 버튼 색과 클릭 여부는 독립이 아닙니다 (관련 있음)")
else:
    print(f"\np값 {p_value:.6f} >= {alpha} → 귀무가설 채택")
    print("→ 버튼 색과 클릭 여부는 독립입니다 (관련 없음)")

In [ ]:
# ┌─────────────────────────────────────────┐
# │  관측빈도 vs 기대빈도 비교 시각화          │
# └─────────────────────────────────────────┘

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 관측빈도
cross.plot(kind='bar', ax=axes[0], color=['steelblue', 'salmon'])
axes[0].set_title('관측빈도 (실제 데이터)')
axes[0].set_ylabel('빈도')
axes[0].set_xlabel('버튼 색')
axes[0].legend(title='클릭')

# 기대빈도
expected_df.plot(kind='bar', ax=axes[1], color=['steelblue', 'salmon'])
axes[1].set_title('기대빈도 (독립 가정 시)')
axes[1].set_ylabel('빈도')
axes[1].set_xlabel('버튼 색')
axes[1].legend(title='클릭')

plt.tight_layout()
plt.savefig('contingency_table.png', dpi=100)
plt.show()

print("관측빈도와 기대빈도의 차이가 클수록 카이제곱 통계량이 커집니다.")

## 3. 카이제곱분포와 p값

카이제곱 통계량이 어느 정도 커야 "유의미한가?"를 판단하려면
**카이제곱분포**를 이해해야 합니다.

- 카이제곱분포는 자유도(df)에 따라 모양이 다릅니다
- 분할표의 자유도: `df = (행 수 - 1) × (열 수 - 1)`
- p값은 카이제곱분포에서 관측값보다 큰 영역의 넓이

> 💡 2×2 분할표의 자유도 = (2-1)×(2-1) = 1

In [ ]:
# ┌─────────────────────────────────────────┐
# │  카이제곱분포와 p값 시각화                 │
# │  관측 χ²값이 기각역에 있는지 확인        │
# └─────────────────────────────────────────┘

# 카이제곱분포 그리기
x = np.arange(0, 15, 0.1)
fig, ax = plt.subplots(figsize=(8, 5))

# df=1 카이제곱분포
ax.plot(x, stats.chi2.pdf(x, df=1), color='black', linewidth=2,
        label='χ²분포 (df=1)')

# 기각역 (p < 0.05)
chi2_critical = stats.chi2.ppf(0.95, df=1)
x_reject = np.arange(chi2_critical, 15, 0.1)
ax.fill_between(x_reject, stats.chi2.pdf(x_reject, df=1),
                alpha=0.3, color='red', label=f'기각역 (χ² > {chi2_critical:.2f})')

# 관측값 표시
ax.axvline(chi2, color='orange', linewidth=2, linestyle='--',
           label=f'관측 χ² = {chi2:.2f}')
ax.axvline(chi2_critical, color='red', linewidth=1, linestyle=':')

ax.set_title('카이제곱분포와 검정 결과')
ax.set_xlabel('χ²')
ax.set_ylabel('확률 밀도')
ax.legend()
plt.tight_layout()
plt.savefig('chi2_distribution.png', dpi=100)
plt.show()

print(f"카이제곱 통계량: {chi2:.4f}")
print(f"기각역 임계값 (α=0.05): {chi2_critical:.4f}")
print(f"p값: {p_value:.6f}")
if chi2 > chi2_critical:
    print(f"\n→ 관측 χ²({chi2:.2f}) > 임계값({chi2_critical:.2f}) → 기각역에 있음!")
else:
    print(f"\n→ 관측 χ²({chi2:.2f}) ≤ 임계값({chi2_critical:.2f}) → 기각역에 없음")

## 4. p값 직접 계산 이해하기

`1 - chi2.cdf(x, df)`로 p값을 직접 계산할 수 있습니다.
이것이 `chi2_contingency()`가 내부적으로 하는 일입니다.

> 💡 카이제곱 검정의 p값은 "관측된 χ²값보다 더 극단적인 값이 나올 확률"입니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  p값 직접 계산                           │
# │  1 - chi2.cdf(x, df)로 계산             │
# │  chi2_contingency()가 내부적으로 하는 일  │
# └─────────────────────────────────────────┘

# 카이제곱 통계량으로 p값 직접 계산
p_manual = 1 - stats.chi2.cdf(x=chi2, df=dof)
print(f"=== p값 직접 계산 ===")
print(f"카이제곱 통계량 χ²: {chi2:.4f}")
print(f"자유도 df: {dof}")
print(f"p값 (직접 계산): {p_manual:.6f}")
print(f"p값 (chi2_contingency): {p_value:.6f}")
print(f"\n→ 두 값이 일치함을 확인")

# Yates 보정이 있는 경우 (2×2 분할표에서만)
result_yates = stats.chi2_contingency(cross, correction=True)
print(f"\n=== Yates 보정 적용 결과 ===")
print(f"χ² (보정 없음): {chi2:.4f}")
print(f"χ² (Yates 보정): {result_yates[0]:.4f}")
print(f"p값 (Yates 보정): {result_yates[1]:.6f}")
print(f"\n💡 Yates 보정은 2×2 분할표에서 더 보수적인 검정 결과를 줍니다.")

## 📋 분할표 검정 요약

| 단계 | 내용 | 함수/코드 |
|------|------|----------|
| 1. 분할표 작성 | 범주별 빈도 정리 | `pd.pivot_table()` |
| 2. 카이제곱 검정 | 독립성 검정 | `stats.chi2_contingency()` |
| 3. 결과 해석 | p < 0.05 → 독립이 아님 | χ² 통계량, p값 |
| 4. 시각화 | 관측 vs 기대빈도 | 막대 그래프 |

> 💡 **핵심**: 카이제곱 검정은 "두 범주형 변수가 독립인가?"를 검정합니다.
> p < 0.05이면 독립이 아니라는 뜻 → 두 변수는 관련이 있습니다.

## 🎯 연습 문제

1. Yates 보정 유무에 따른 χ² 통계량과 p값의 차이를 확인하고, 언제 보정을 사용하는지 설명하세요.
2. 새로운 분할표를 만들어 카이제곱 검정을 수행하세요 (예: 성별과 선호 색상).
3. 자유도가 2, 4, 6인 카이제곱분포를 한 그래프에 그리세요.
4. 분할표의 빈도를 2배로 늘렸을 때 검정 결과가 어떻게 변하는지 확인하세요.